In [ ]:
# 1. Instalação
# pip install -q bertopic[visualization] sentence-transformers umap-learn hdbscan
# pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [1]:
# 2. Importações
import pandas as pd
import torch

from sentence_transformers import SentenceTransformer
from bertopic import BERTopic

from umap import UMAP
import hdbscan

I0000 00:00:1774869034.821405    2141 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774869035.754165    2141 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774869037.839771    2141 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
# 3. Verificar GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
#Esperado ==> Device: cuda

Device: cuda


In [3]:
#4. Carregar dados
df = pd.read_csv("/home/paulo/projeto_tech_05/complaints_sentiment_retail_banking_negative.csv")

# supondo coluna chamada 'narrative' com os textos.
docs = df['narrative'].dropna().tolist()

len(docs)

153

In [4]:
# 5. (Opcional) Limpeza leve
import re

def limpar(texto):
    texto = texto.lower()
    texto = re.sub(r'\d+', '', texto)
    texto = re.sub(r'[^\w\s]', '', texto)
    return texto

docs = [limpar(t) for t in docs]

In [5]:
# 6. Embeddings na GPU
model = SentenceTransformer(
    'paraphrase-multilingual-MiniLM-L12-v2',
    device=device
)

embeddings = model.encode(
    docs,
    batch_size=128,          # ajuste conforme VRAM
    show_progress_bar=True,
    device=0  # Specify GPU device (0 = GPU device, -1 = CPU)
)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

In [6]:
# 7. Redução de dimensionalidade (UMAP)
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric='cosine'
)

In [7]:
# 8. Clusterização (HDBSCAN)
hdbscan_model = hdbscan.HDBSCAN(
    min_cluster_size=30,
    metric='euclidean',
    cluster_selection_method='eom'
)

In [ ]:
# 9. Modelo BERTopic
# 9. Modelo BERTopic
topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    calculate_probabilities=True,
    verbose=True
)

topics, probs = topic_model.fit_transform(docs, embeddings)

NameError: name 'topic_model' is not defined

In [ ]:
# 10. Treinar modelo
topic_info = topic_model.get_topic_info()
topic_info.head(10)

In [ ]:
# 11. Ver os principais temas
topic_info = topic_model.get_topic_info()
topic_info.head(10)

In [ ]:
# 12. Ver palavras de um tópico
topic_model.get_topic(0)

In [ ]:
# 13. Ver exemplos de reclamações por tema
topic_model.get_representative_docs(0)

In [ ]:
# 14. Visualização (muito útil)
topic_model.visualize_topics()

In [ ]:
topic_model.visualize_barchart()

In [ ]:
#15. Nomear temas com LLM (opcional)
topic_model.update_topics(docs)

In [ ]:
# Otimizações para RTX 3060
# Ajustes recomendados:
embeddings = model.encode(
    docs,
    batch_size=64,   # aumente se couber na VRAM
    normalize_embeddings=True
)

In [ ]:
# Resultado esperado

# Você vai obter clusters como:

# “atraso entrega”
# “cobrança indevida”
# “cancelamento difícil”
# “produto com defeito”

# Cada um com:

# quantidade de ocorrências
# palavras-chave
# exemplos reais

In [ ]:
# Extensão prática (produção)

# Depois disso, você pode:

# Criar dashboard (Power BI / Streamlit)
# Monitorar evolução de temas no tempo
# Detectar novos problemas automaticamente